In [3]:
# Work around a TRL 0.24.0 + Transformers 5.5.0 bug where prompt-only apply_chat_template() returns a BatchEncoding, causing TRL to miscompute the prompt length (e.g., len(prompt_ids) == 2) and incorrectly mask only the first few prompt tokens.
# SFT caompatibe != TRL 0.24.0 & Transformers 5.5 -> prompt-only apply_chat_template() error reported
%pip install --upgrade --no-cache-dir transformers trl datasets peft accelerate bitsandbytes safetensors
!pip install --upgrade "torchao>0.16.0" # PEFT requirement

In [4]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Colab_Notebooks') # change directory to the current working directory

Mounted at /content/drive


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import pandas as pd
from datasets import Dataset

DPO_TRAIN_DATA_LOAD_PATH = "data_preference_train_train_only/bridge_2tage__with_reference_solution_more_models/response_pairs_df.csv"
TEST_DATA_LOAD_PATH = "train_test_split/test_stepverify_labeled_0.9.json"

BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

SFT_ADAPTER_PATH = "llama3-8b-instruct-sft-adapter" # Adapter name # DPO (pi_theat, pi_ref)

DPO_ADAPTER_PATH_05 = "llama3-8b-instruct-dpo-adapter--threshold_05" # DPO_ADAPTER_SAVE_PATH
DPO_ADAPTER_PATH_03 = "llama3-8b-instruct-dpo-adapter--threshold_03" # DPO_ADAPTER_SAVE_PATH
DPO_ADAPTER_PATH_01 = "llama3-8b-instruct-dpo-adapter--threshold_01" # DPO_ADAPTER_SAVE_PATH

DRIVE_ROOT_DIR = "/content/drive/My Drive/Colab_Notebooks" if IN_COLAB else "" # current notebook directory in the google drive

DPO_MODEL_DIR_05 = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_PATH_05)    # save in the ADAPTER directory in the google drive  # DPO_DRIVE_MODEL_DIR
DPO_MODEL_DIR_03 = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_PATH_03)    # save in the ADAPTER directory in the google drive  # DPO_DRIVE_MODEL_DIR
DPO_MODEL_DIR_01 = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_PATH_01)    # save in the ADAPTER directory in the google drive  # DPO_DRIVE_MODEL_DIR

SFT_DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT_DIR, SFT_ADAPTER_PATH)

SEED = 42

In [6]:
# from huggingface_hub import notebook_login
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
login(token=hf_token )

In [7]:
SYSTEM_TEMPLATE = """You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.
"""

USER_TEMPLATE = """
### Problem:
{problem}

### Student's response:
{student}
"""


def make_dpo_test_dataset(data):
    dataset = Dataset.from_list([
        {
            # Metadata
            "topic": row["topic"],
            "problem": row["problem"],
            "student_mistake": row["student_mistake"],
            "reference_solution": row["reference_solution"],
            "error_category": row["error_category"],
            "error_description": row["error_description"],
            "student_correct_response": row["student_correct_response"],

            "prompt": [
                {
                    "role": "system",
                    "content": SYSTEM_TEMPLATE,
                },
                {
                    "role": "user",
                    "content": USER_TEMPLATE.format(
                        problem=str(row["problem"]),
                        student=str(row["student_mistake"]),
                    ),
                },
            ],
            "completion": [
              {
                  "role": "assistant",
                  "content": "(" + str(row['dialog_history'][0]['pedagogy']) + ")" + str(row['dialog_history'][0]['text']),
              }
          ],
        }
        for row in data
    ])

    return dataset

In [8]:
test_data = json.load(open(TEST_DATA_LOAD_PATH, "r"))
# test_data = pd.DataFrame(test_data).rename(columns={"student_incorrect_solution": "student_mistake"}).to_dict(orient="records") # change the key name to match pref_data_format
for row in test_data:
    row["student_mistake"] = row.pop("student_incorrect_solution")
test_ds = make_dpo_test_dataset(test_data)

In [9]:
test_ds[0]

{'topic': 'Math Word Problem',
 'problem': 'lori owns a carsharing company. there are three red cars and two white cars available to rent. renting the white car costs 2 for every minute and the red car 3 for every minute. all cars were rented for 3 hours. how much money did lori earn?',
 'student_mistake': "'renting the white car for 3 hours 180 minutes costs 2 x 180 360.', 'renting the red car for 3 hours 180 minutes costs 3 x 180 540 per car.', 'lori earned 3 x 540 1620 for the three red cars.', 'lori earned a total of 1620 360 1980 for all the cars.', ' 1980'",
 'reference_solution': "Both white cars were rented for 2 * 2 = $4 for every minute.\nAll three red cars were rented for 3 * 3 = $9 for every minute.\nSo all Lori's cars were rented for 4 + 9 = $13 for every minute.\nThree hours are 180 minutes, so Lori earned 13 * 180 = $2340 for renting all the cars.\n 2340",
 'error_category': 'misunderstanding_of_a_question',
 'error_description': None,
 'student_correct_response': "Lori 

#### Generate responses per model
- the base model is initially loaded and then each adapter is alternatively attached to the base model.

In [10]:
############
import torch
from transformers import pipeline
from tqdm.auto import tqdm
from copy import deepcopy
from transformers.utils import logging

logging.set_verbosity_error() # turn off warning ( both max_new_tokens and max_length are set .... )

def get_base_model_and_tokenizer():

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        device_map={"": 0},
    )

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        clean_up_tokenization_spaces=False,
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()

    return model, tokenizer


def check_adapter(model):
    print(f"Checking adapter status..")
    if hasattr(model, "peft_config") and model.peft_config:
        print(f"  Loaded adapters ( {list(model.peft_config.keys())} )")
        print(f"  Active adapter ( {model.active_adapters()} )")
        assert model.active_adapters(), "Adapter is loaded but no adapter is active."
    else:
        print(f"  No adapter loaded")



def generate_responses(model, tokenizer, dataset, batch_size=64):

    check_adapter(model) # check if model has an adapter

    prompts = [
        tokenizer.apply_chat_template(
            data["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )
        for data in dataset
    ]

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text=False,
    )

    outputs = []

    print("Generating responses..")
    print(
        f"  Total Rows: {len(prompts)} | "
        f"Batch Size: {batch_size} | "
        f"Total Batches: {(len(prompts) + batch_size - 1) // batch_size} |\n"
    )

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_outputs = pipe(
            prompts[i:i + batch_size],
            do_sample=False,
            clean_up_tokenization_spaces=False,
        )

        outputs.extend(batch_outputs)

    results = {
        "student_mistake": [ row["student_mistake"] for row in dataset ],
        "topic": [ row["topic"] for row in dataset ],
        "problem": [ row["problem"] for row in dataset ],
        "error_category": [ row["error_category"] for row in dataset ],
        "error_description": [ row["error_description"] for row in dataset ],
        "reference_solution": [ row["reference_solution"] for row in dataset ],
        "student_correct_response": [ row["student_correct_response"] for row in dataset ],
        "ground_truth": [ row["completion"][0]["content"] for row in dataset ],
        "prompts": prompts,
        "llm_response": [ output[0]["generated_text"].strip() for output in outputs ],

    }

    print("=" * 100)

    return results



def generate_all_responses(dataset, adapter_dirs):
    base_model, base_tokenizer = get_base_model_and_tokenizer()

    results = {}

    # Base model
    print("[ Current Model : base ]")
    results["base"] = generate_responses( base_model, base_tokenizer, dataset)

    # Fine-tuned models
    for model_name, adapter_dir in adapter_dirs.items():
        print(f"[ Current Model : {model_name} ]")

        # Load adapter using the same temporary name
        base_model.load_adapter(adapter_dir, is_trainable=False )

        base_model.set_adapter("default") # activate the adapter. default name = "default" https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/peft?utm_source=chatgpt.com

        # Generate
        results[model_name] = generate_responses(base_model, base_tokenizer, dataset,)

        # Remove adapter before loading the next one
        base_model.delete_adapter("default")

    return results

In [11]:
adapter_dirs = {
    "sft": SFT_DRIVE_MODEL_DIR,
    "dpo_01": DPO_MODEL_DIR_01,
    "dpo_03": DPO_MODEL_DIR_03,
    "dpo_05": DPO_MODEL_DIR_05,
}

model_responses = generate_all_responses(test_ds, adapter_dirs,)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Generating:   0%|          | 0/5 [00:00<?, ?it/s]

[ Current Model : dpo_05 ]


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Checking adapter status..
  Loaded adapters ( ['default'] )
  Active adapter ( ['default'] )
Generating responses..
  Total Rows: 298 | Batch Size: 64 | Total Batches: 5 |



Generating:   0%|          | 0/5 [00:00<?, ?it/s]

#### Save generated responses

In [12]:
import json
import pandas as pd

with open("model_responses_sft_dpo__raw.json", "w", encoding="utf-8") as f:
    json.dump(model_responses, f, ensure_ascii=False, indent=2, )

In [13]:
import pandas as pd

pd.DataFrame(model_responses)

,base,sft,dpo_01,dpo_03,dpo_05
student_mistake,['renting the white car for 3 hours 180 minute...,['renting the white car for 3 hours 180 minute...,['renting the white car for 3 hours 180 minute...,['renting the white car for 3 hours 180 minute...,['renting the white car for 3 hours 180 minute...
topic,"[Math Word Problem, Math Word Problem, Math Wo...","[Math Word Problem, Math Word Problem, Math Wo...","[Math Word Problem, Math Word Problem, Math Wo...","[Math Word Problem, Math Word Problem, Math Wo...","[Math Word Problem, Math Word Problem, Math Wo..."
problem,[lori owns a carsharing company. there are thr...,[lori owns a carsharing company. there are thr...,[lori owns a carsharing company. there are thr...,[lori owns a carsharing company. there are thr...,[lori owns a carsharing company. there are thr...
error_category,"[misunderstanding_of_a_question, extra_quantit...","[misunderstanding_of_a_question, extra_quantit...","[misunderstanding_of_a_question, extra_quantit...","[misunderstanding_of_a_question, extra_quantit...","[misunderstanding_of_a_question, extra_quantit..."
error_description,"[None, Added quantity, None, student calculate...","[None, Added quantity, None, student calculate...","[None, Added quantity, None, student calculate...","[None, Added quantity, None, student calculate...","[None, Added quantity, None, student calculate..."
reference_solution,[Both white cars were rented for 2 * 2 = $4 fo...,[Both white cars were rented for 2 * 2 = $4 fo...,[Both white cars were rented for 2 * 2 = $4 fo...,[Both white cars were rented for 2 * 2 = $4 fo...,[Both white cars were rented for 2 * 2 = $4 fo...
student_correct_response,[Lori had 3 red cars and 2 white cars availabl...,[Lori had 3 red cars and 2 white cars availabl...,[Lori had 3 red cars and 2 white cars availabl...,[Lori had 3 red cars and 2 white cars availabl...,[Lori had 3 red cars and 2 white cars availabl...
ground_truth,"[(generic)hi brenda, could you please walk me ...","[(generic)hi brenda, could you please walk me ...","[(generic)hi brenda, could you please walk me ...","[(generic)hi brenda, could you please walk me ...","[(generic)hi brenda, could you please walk me ..."
prompts,[<|begin_of_text|><|start_header_id|>system<|e...,[<|begin_of_text|><|start_header_id|>system<|e...,[<|begin_of_text|><|start_header_id|>system<|e...,[<|begin_of_text|><|start_header_id|>system<|e...,[<|begin_of_text|><|start_header_id|>system<|e...
llm_response,[That's a great start! You're correct that the...,"[how many cars are there in total?, if it take...","[great work!, if bert uses up a pencil every t...","[great!, if it takes 1050 words to use up a pe...","[great!, if bert uses up a pencil every two we..."


In [14]:
model_responses = json.load(open("model_responses_sft_dpo__raw.json", "r"))

In [15]:
# common data
topics = model_responses['base']['topic']
problems = model_responses['base']['problem']
error_categories = model_responses['base']['error_category']
error_descriptions = model_responses['base']['error_description']
reference_solutions = model_responses['base']['reference_solution']
student_correct_responses = model_responses['base']['student_correct_response']
student_mistakes = model_responses['base']['student_mistake']
ground_truth = model_responses['base']['ground_truth']
prompts = model_responses['base']['prompts']

# responses
base_responses = model_responses['base']['llm_response']
sft_responses = model_responses['sft']['llm_response']
dpo_01_responses = model_responses['dpo_01']['llm_response']
dpo_03_responses = model_responses['dpo_03']['llm_response']
dpo_05_responses = model_responses['dpo_05']['llm_response']

model_responses__sorted__df = pd.DataFrame({
    "topic": topics,
    "problem": problems,
    "error_category": error_categories,
    "error_description": error_descriptions,
    "reference_solution": reference_solutions,
    "student_correct_response": student_correct_responses,
    "student_mistake": student_mistakes,
    "original_tutor_response": ground_truth,
    "prompt": prompts,

    "base": base_responses,
    "sft": sft_responses,
    "dpo_01": dpo_01_responses,
    "dpo_03": dpo_03_responses,
    "dpo_05": dpo_05_responses,
})

model_responses__sorted__df.to_csv("model_responses_sft_dpo__sorted_df.csv")

In [16]:
for idx, row in model_responses__sorted__df.iterrows():

    print(f"Index: {idx}")

    print(f"prompt: {row['prompt']}")
    print("="*100)
    for column, value in row.items():
        if column == "sft":
            print("SFT")
            print(value)
        elif column == "dpo_01":
            print("DPO_01")
            print(value)
        elif column == "dpo_03":
            print("DPO_03")
            print(value)
        elif column == "dpo_05":
            print("DPO_05")
            print(value)


    print("=" * 100)

Index: 0
prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Problem:
lori owns a carsharing company. there are three red cars and two white cars available to rent. renting the white car costs 2 for every minute and the red car 3 for every minute. all cars were rented for 3 hours. how much money did lori earn?

### Student's response:
'renting the white car for 3 hours 180 minutes costs 2 x 180 360.', 'renting the red car for 3 hours 180 minutes costs 3 x 180 540 per car.', 'lori earned 3 x 540 1620 for the three red cars.', 'l

---
previous code
- short of memory ( requires 90GB + GPU memory )

In [17]:

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel, PeftConfig
# import torch

# def get_model_and_tokenizer(model_dir):
#     """
#     Load model and tokenizer conveniently for both base and peft/dpo model_dir.
#     If model_dir contains a PEFT adapter, loads it on top of the correct base model.
#     Otherwise, loads a vanilla base model from model_dir.
#     This attempts to infer PEFT vs base automatically by trying to read PEFT config.
#     """

#     # The PEFT adapter directory should contain "adapter_config.json" (or similar for LORA/DPO)
#     is_peft = os.path.isfile(os.path.join(model_dir, "adapter_config.json"))

#     if is_peft:
#         from peft import PeftConfig
#         # Load peft config and extract base model
#         peft_config = PeftConfig.from_pretrained(model_dir)
#         base_model_id = peft_config.base_model_name_or_path
#         # Load base model
#         base_model = AutoModelForCausalLM.from_pretrained(
#             base_model_id,
#             dtype=torch.bfloat16 if ( torch.cuda.is_available() and torch.cuda.is_bf16_supported() ) else torch.float16,
#             device_map={"": 0},
#         )

#         # Apply PEFT adapter (DPO, LORA, etc.)
#         model = PeftModel.from_pretrained(
#             base_model,
#             model_dir,
#             is_trainable=False,
#         )

#         tokenizer = AutoTokenizer.from_pretrained(
#             model_dir,
#             clean_up_tokenization_spaces=False,
#         )
#         if tokenizer.pad_token is None:
#             tokenizer.pad_token = tokenizer.eos_token

#         tokenizer.padding_side = "left"
#         model.config.pad_token_id = tokenizer.pad_token_id

#     else:
#         # Load base model and tokenizer only (no adapters)
#         model = AutoModelForCausalLM.from_pretrained(
#             model_dir,
#             dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16dtype,
#             device_map={"": 0},
#         )
#         tokenizer = AutoTokenizer.from_pretrained(
#             model_dir,
#             clean_up_tokenization_spaces=False,
#         )
#         if tokenizer.pad_token is None:
#             tokenizer.pad_token = tokenizer.eos_token
#         tokenizer.padding_side = "left"
#         model.config.pad_token_id = tokenizer.pad_token_id

#     return model, tokenizer


In [18]:
# dpo_model_01, dpo_tokenizer_01 = get_model_and_tokenizer(DPO_MODEL_DIR_01)
# dpo_model_03, dpo_tokenizer_03 = get_model_and_tokenizer(DPO_MODEL_DIR_03)
# dpo_model_05, dpo_tokenizer_05 = get_model_and_tokenizer(DPO_MODEL_DIR_05)
# sft_model, sft_tokenizer = get_model_and_tokenizer(SFT_DRIVE_MODEL_DIR)
# base_model, base_tokenizer = get_model_and_tokenizer(BASE_MODEL_ID)

In [19]:
# from transformers import pipeline

# def generate_responses(model, tokenizer, dataset):

#     prompts = [
#         tokenizer.apply_chat_template(
#             data["prompt"],
#             tokenize=False,
#             add_generation_prompt=True,
#         )
#         for data in dataset
#     ]

#     pipe = pipeline(
#         task="text-generation",
#         model=model,
#         tokenizer=tokenizer,
#         return_full_text=False,
#     )

#     outputs = pipe(
#         prompts,
#         max_new_tokens=128,
#         do_sample=False,
#         clean_up_tokenization_spaces=False,
#     )

#     return [
#         output[0]["generated_text"].strip() for output in outputs
#     ]


# base_responses = generate_responses(
#     base_model,
#     base_tokenizer,
#     test_ds,
# )

# # sft_responses = generate_responses(
# #     sft_model,
# #     sft_tokenizer,
# #     test_ds,
# # )

# # dpo_responses_01 = generate_responses(
# #     dpo_model_01,
# #     dpo_tokenizer_01,
# #     test_ds,
# # )

# # dpo_responses_03 = generate_responses(
# #     dpo_model_01,
# #     dpo_tokenizer_01,
# #     test_ds,
# # )

# # dpo_responses_05 = generate_responses(
# #     dpo_model_01,
# #     dpo_tokenizer_01,
# #     test_ds,
# # )